In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
df = pd.read_csv("../data/raw/ncr_ride_bookings.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (150000, 21)


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  str    
 1   Time                               150000 non-null  str    
 2   Booking ID                         150000 non-null  str    
 3   Booking Status                     150000 non-null  str    
 4   Customer ID                        150000 non-null  str    
 5   Vehicle Type                       150000 non-null  str    
 6   Pickup Location                    150000 non-null  str    
 7   Drop Location                      150000 non-null  str    
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  10500 non-null 

In [5]:
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})

missing.sort_values("Missing %", ascending=False)

,Missing Count,Missing %
Incomplete Rides Reason,141000,94.0
Incomplete Rides,141000,94.0
Cancelled Rides by Customer,139500,93.0
Reason for cancelling by Customer,139500,93.0
Driver Cancellation Reason,123000,82.0
Cancelled Rides by Driver,123000,82.0
Customer Rating,57000,38.0
Driver Ratings,57000,38.0
Ride Distance,48000,32.0
Booking Value,48000,32.0


In [6]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Booking IDs:", df["Booking ID"].duplicated().sum())

Duplicate rows: 0
Duplicate Booking IDs: 1233


In [7]:
df["Booking Status"].value_counts(dropna=False)

Booking Status
Completed                93000
Cancelled by Driver      27000
No Driver Found          10500
Cancelled by Customer    10500
Incomplete                9000
Name: count, dtype: int64

In [8]:
df["Booking Status"].value_counts(normalize=True).mul(100).round(2)

Booking Status
Completed                62.0
Cancelled by Driver      18.0
No Driver Found           7.0
Cancelled by Customer     7.0
Incomplete                6.0
Name: proportion, dtype: float64

In [9]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n===== {col} =====")
    print(df[col].nunique())
    print(df[col].dropna().unique()[:20])

C:\Users\Admin\AppData\Local\Temp\ipykernel_1180\1333925406.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns



===== Date =====
365
<StringArray>
['2024-03-23', '2024-11-29', '2024-08-23', '2024-10-21', '2024-09-16',
 '2024-02-06', '2024-06-17', '2024-03-19', '2024-09-14', '2024-12-16',
 '2024-06-14', '2024-09-18', '2024-06-25', '2024-09-11', '2024-10-18',
 '2024-06-07', '2024-07-01', '2024-12-15', '2024-11-24', '2024-05-24']
Length: 20, dtype: str

===== Time =====
62910
<StringArray>
['12:29:38', '18:01:39', '08:56:10', '17:17:25', '22:08:00', '09:44:56',
 '15:45:58', '17:37:37', '12:49:09', '19:06:48', '16:24:12', '08:09:38',
 '22:44:15', '19:29:39', '18:28:53', '15:05:35', '10:51:16', '15:08:25',
 '09:07:10', '19:53:57']
Length: 20, dtype: str

===== Booking ID =====
148767
<StringArray>
['"CNR5884300"', '"CNR1326809"', '"CNR8494506"', '"CNR8906825"',
 '"CNR1950162"', '"CNR4096693"', '"CNR2002539"', '"CNR6568000"',
 '"CNR4510807"', '"CNR7721892"', '"CNR9070334"', '"CNR9551927"',
 '"CNR4386945"', '"CNR2987763"', '"CNR8962232"', '"CNR2390352"',
 '"CNR3221338"', '"CNR6739317"', '"CNR6126048"'

In [10]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Avg VTAT,139500.0,8.456352,3.773564,2.0,5.30,8.30,11.30,20.0
Avg CTAT,102000.0,29.149636,8.902577,10.0,21.60,28.80,36.80,45.0
Cancelled Rides by Customer,10500.0,1.000000,0.000000,1.0,1.00,1.00,1.00,1.0
Cancelled Rides by Driver,27000.0,1.000000,0.000000,1.0,1.00,1.00,1.00,1.0
Incomplete Rides,9000.0,1.000000,0.000000,1.0,1.00,1.00,1.00,1.0
Booking Value,102000.0,508.295912,395.805774,50.0,234.00,414.00,689.00,4277.0
Ride Distance,102000.0,24.637012,14.002138,1.0,12.46,23.72,36.82,50.0
Driver Ratings,93000.0,4.230992,0.436871,3.0,4.10,4.30,4.60,5.0
Customer Rating,93000.0,4.404584,0.437819,3.0,4.20,4.50,4.80,5.0


In [11]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.strip('"')

C:\Users\Admin\AppData\Local\Temp\ipykernel_1180\2339903438.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [12]:
df[["Booking ID", "Customer ID"]].head()

,Booking ID,Customer ID
0,CNR5884300,CID1982111
1,CNR1326809,CID4604802
2,CNR8494506,CID9202816
3,CNR8906825,CID2610914
4,CNR1950162,CID9933542


In [13]:
df["DateTime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce"
)

In [14]:
df["Year"] = df["DateTime"].dt.year
df["Month"] = df["DateTime"].dt.month
df["Month_Name"] = df["DateTime"].dt.month_name()
df["Day"] = df["DateTime"].dt.day
df["Day_Name"] = df["DateTime"].dt.day_name()
df["Hour"] = df["DateTime"].dt.hour
df["Day_of_Week"] = df["DateTime"].dt.dayofweek

In [15]:
df["Is_Weekend"] = df["Day_of_Week"] >= 5

In [16]:
df["Is_Completed"] = (
    df["Booking Status"] == "Completed"
).astype(int)

In [17]:
df["Is_Cancelled"] = (
    df["Booking Status"].str.contains(
        "Cancelled",
        case=False,
        na=False
    )
).astype(int)

In [18]:
df["Is_Successful"] = (
    df["Booking Status"] == "Completed"
).astype(int)

In [19]:
df["Revenue"] = np.where(
    df["Booking Status"] == "Completed",
    df["Booking Value"],
    0
)

In [20]:
df["Revenue"] = np.where(
    df["Booking Status"] == "Completed",
    df["Booking Value"],
    0
)

In [21]:
output_path = "../data/processed/uberpulse_cleaned.csv"

df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")

Saved cleaned dataset to: ../data/processed/uberpulse_cleaned.csv
